# Shea Homes Customer Review Analysis
### Notebook: `02_data_evaluation`

> Run `01_setup_and_summary_stats.ipynb` first, or execute the setup cell below.

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from scipy import stats
import re
import warnings
warnings.filterwarnings("ignore")

# nlp tools
import nltk
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from textblob import TextBlob
from wordcloud import WordCloud
import spacy

#ml
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import Pipeline

# visual style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
SHEA_BLUE = "#1a5276"
SHEA_GOLD = "#d4a843"
SHEA_PALETTE = ["#c0392b", "#e67e22", "#f1c40f", "#27ae60", "#1a5276"]

# keep apostrophes so contractions like "don't" stay as one token
TOKEN_PATTERN = r"(?u)\b\w[\w']+\b"

In [32]:
# load the dataset
df = pd.read_csv("../shea_homes_reviews.csv", encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# derived columns
df["word_count"] = df["review_text"].apply(lambda x: len(str(x).split()))
df["char_count"] = df["review_text"].apply(lambda x: len(str(x)))
df["state"] = df["location"].str.extract(r",\s*([A-Z]{2})$")
df["year"] = df["date"].dt.year
df["year_month"] = df["date"].dt.to_period("M")
df["quarter"] = df["date"].dt.to_period("Q")

print(f"{len(df):,} reviews")
print(f"Date range: {df['date'].min().strftime('%B %Y')} to {df['date'].max().strftime('%B %Y')}")
df.head(3)

2,052 reviews
Date range: September 2020 to April 2026


,title,reviewer_name,verified_homebuyer,date,location,review_text,total_score,quality,trustworthiness,value,responsiveness,word_count,char_count,state,year,year_month,quarter
0,Good overall,Constance G.,Yes,2026-04-01,"OCALA, FL",A little overwhelming for me as a single elder...,5,5,5,5,5,86,434,FL,2026,2026-04,2026Q2
1,"Excellent, knowledgeable and responsive",Emily V.,Yes,2026-03-31,"Littleton, CO",Our experience exceeded our expectations. The ...,5,5,5,5,5,32,218,CO,2026,2026-03,2026Q1
2,Overall great experience!,Lynn S.,Yes,2026-03-26,"Kuna, ID",This is our second Shea home! Our whole proces...,5,5,5,5,5,83,516,ID,2026,2026-03,2026Q1


---
# Part 2: Data Evaluation

Before conducting analysis, the dataset must first be evaluated for quality, representativeness, and potential limitations. Key considerations include whether 2,039 reviews constitute an adequate sample size, and whether any biases or gaps in coverage may affect the findings. This section provides a transparent assessment of the dataset's strengths and limitations prior to deriving recommendations.

## 2.1 Suitability for Business Questions

Core business question: What do customer reviews show about the Shea Homes homebuying experience, and which aspects of that experience represent the largest opportunity for improvement?


This dataset is well-suited to answer this because:
- Every review is from a verified homebuyer, meaning this is real customer feedback
- Reviews contain free-text responses in addition to star ratings
- Five separate rating dimensions (Quality, Trustworthiness, Value, Responsiveness) let us pinpoint which aspect of the experience is driving satisfaction or dissatisfaction
- Geographic and temporal data let us track performance across markets and over time

## 2.2 Sample Size Assessment

This section evaluates whether the dataset is large enough to support reliable conclusions. It summarizes the total number of reviews, the average rating, and how reviews are distributed across states.

In [36]:
n = len(df)
mean_score = df["total_score"].mean()
std_score = df["total_score"].std()

# 95% confidence interval for the mean
margin_of_error = 1.96 * (std_score / np.sqrt(n))
ci_low = mean_score - margin_of_error
ci_high = mean_score + margin_of_error

print("SAMPLE SIZE")
print("=" * 50)
print(f"  Sample size:              {n:,} reviews")
print(f"  Mean total score:         {mean_score:.3f}")
print(f"  Standard deviation:       {std_score:.3f}")
print(f"  95% Confidence Interval:  [{ci_low:.3f}, {ci_high:.3f}]")
print(f"  Margin of error:          ±{margin_of_error:.3f} stars")
print()

# per-state sample sizes
print(f"\n  Per-state sample sizes:")
for state, count in df["state"].value_counts().items():
    print(f"    {state}: {count:>4} reviews")

SAMPLE SIZE
  Sample size:              2,052 reviews
  Mean total score:         4.212
  Standard deviation:       1.110
  95% Confidence Interval:  [4.164, 4.260]
  Margin of error:          ±0.048 stars


  Per-state sample sizes:
    CA:  689 reviews
    AZ:  533 reviews
    CO:  180 reviews
    TX:  160 reviews
    NV:  104 reviews
    NC:   98 reviews
    WA:   90 reviews
    VA:   80 reviews
    FL:   75 reviews
    ID:   35 reviews
    SC:    8 reviews


The dataset contains 2,039 reviews with an average rating of 4.21. To estimate how precise this average is, we calculate a 95% confidence interval, which represents the range where the true average customer rating is likely to fall. The interval of 4.162 to 4.259 indicates that the estimated average rating is statistically stable. Review counts vary across states. Arizona and California account for the largest share of feedback, while several states have smaller samples. Locations with fewer reviews provide less statistical certainty and should be interpreted cautiously.


## 2.3 Potential Biases

Online review datasets often contain structural biases that can influence how results should be interpreted. This section identifies several common sources of bias within the dataset and explains how they may affect the conclusions drawn from the analysis.


Here are the biases we should keep in mind:

| Bias | Description | Impact |
|------|-------------|--------|
| **Self-selection bias** | Customers with strong opinions (very happy or very unhappy) are more likely to leave a review. Customers with "just okay" experiences may not bother. | May overrepresent extreme opinions |
| **Geographic concentration** | CA and AZ account for ~60% of reviews. Insights may not generalize equally to smaller markets like ID or SC. | State-level conclusions should note sample sizes |
| **Temporal skew** | 2021–2022 had peak review volume (post-COVID housing boom). Recent years have fewer reviews. | Trends may reflect market conditions, not just Shea's performance |
| **Platform bias** | TrustBuilder® is a builder-partnered review platform. Reviews may skew more positive compared to an independent site like Yelp or BBB. | Overall sentiment may be inflated |

The most significant is platform bias. Because the reviews originate from a builder-partnered platform, overall ratings may skew more positive than reviews found on independent consumer sites. As a result, the negative reviews that do appear in the dataset are particularly informative. They represent customers whose dissatisfaction was strong enough to be expressed despite the generally positive environment of the platform.
